# Student Performance Prediction (ML)

**Task:** Predict student exam result / grade category

**Dataset:** UCI Student Performance Data Set (Mathematics)

**Deliverables:**
- Data preprocessing & EDA
- Classification models: Logistic Regression, Random Forest, XGBoost
- Feature importance analysis
- Trained model + Accuracy + Confusion Matrix

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import xgboost as xgb
import joblib
import warnings
warnings.filterwarnings('ignore')

# Load data (semicolon-separated)
df = pd.read_csv('student-mat.csv', sep=';')

# Ensure grade columns are numeric (CSV may have quoted strings)
for col in ['G1', 'G2', 'G3']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('Shape:', df.shape)
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
print('\n=== Data Types & Non-null ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum().sum(), '(None)')

In [ ]:
# Grade distribution (G3 = final grade, target)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, col in enumerate(['G1', 'G2', 'G3']):
    axes[i].hist(df[col], bins=21, edgecolor='black', alpha=0.7)
    axes[i].set_title(f'{col} Grade Distribution')
    axes[i].set_xlabel('Grade')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix (numeric features)
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=False, cmap='RdYlBu_r', center=0, square=True)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with target G3
target_corr = df[numeric_cols].corr()['G3'].drop('G3').sort_values(ascending=False)
target_corr.plot(kind='barh', figsize=(8, 6))
plt.title('Feature Correlation with Final Grade (G3)')
plt.xlabel('Correlation')
plt.tight_layout()
plt.show()

## 3. Data Preprocessing

In [ ]:
# Create grade category from G3 (0-20 scale, 10 is passing)
# Fail: <10, Pass: 10-13, Good: 14-16, Excellent: 17+
def grade_to_category(g):
    if g < 10: return 'Fail'
    elif g < 14: return 'Pass'
    elif g < 17: return 'Good'
    else: return 'Excellent'

df['grade_category'] = df['G3'].apply(grade_to_category)
print('Grade Category Distribution:')
print(df['grade_category'].value_counts())

In [ ]:
# Encode categorical variables
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
categorical_cols = [c for c in categorical_cols if c != 'grade_category']  # exclude target

df_encoded = df.copy()
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
    label_encoders[col] = le

# Features: exclude G1, G2, G3 and grade_category (target)
feature_cols = [c for c in df_encoded.columns if c not in ['G1', 'G2', 'G3', 'grade_category']]
X = df_encoded[feature_cols]
y = df_encoded['grade_category']

# Label encode target
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

print('Features:', len(feature_cols))
print('Classes:', le_target.classes_)

In [ ]:
# Train-test split & scaling
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Train size:', len(X_train), '| Test size:', len(X_test))

## 4. Classification Models

In [ ]:
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    cm = confusion_matrix(y_te, y_pred)
    print(f'\n=== {name} ===')
    print(f'Accuracy: {acc:.4f}')
    print(f'Confusion Matrix:\n{cm}')
    print(f'\nClassification Report:\n{classification_report(y_te, y_pred, target_names=le_target.classes_)}')
    return model, acc, cm

In [ ]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr_model, lr_acc, lr_cm = evaluate_model('Logistic Regression', lr, X_train_scaled, X_test_scaled, y_train, y_test)

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model, rf_acc, rf_cm = evaluate_model('Random Forest', rf, X_train, X_test, y_train, y_test)

In [ ]:
# XGBoost
xgb_clf = xgb.XGBClassifier(n_estimators=100, random_state=42, verbosity=0)
xgb_model, xgb_acc, xgb_cm = evaluate_model('XGBoost', xgb_clf, X_train, X_test, y_train, y_test)

## 5. Feature Importance Analysis

In [ ]:
# Random Forest & XGBoost feature importance
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

rf_imp = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=True)
rf_imp.tail(15).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Random Forest - Top 15 Feature Importance')

xgb_imp = pd.Series(xgb_model.feature_importances_, index=feature_cols).sort_values(ascending=True)
xgb_imp.tail(15).plot(kind='barh', ax=axes[1], color='darkgreen')
axes[1].set_title('XGBoost - Top 15 Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Logistic Regression coefficients (for scaled features)
lr_coef = pd.Series(np.abs(lr_model.coef_).mean(axis=0), index=feature_cols).sort_values(ascending=False)
lr_coef.head(15).plot(kind='barh', figsize=(8, 6), color='coral')
plt.title('Logistic Regression - Feature |Coefficients| (mean)')
plt.tight_layout()
plt.show()

## 6. Summary & Model Persistence

In [ ]:
# Accuracy comparison
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Accuracy': [lr_acc, rf_acc, xgb_acc]
})
print(results.to_string(index=False))

# Best model (XGBoost typically performs best)
best_idx = np.argmax([lr_acc, rf_acc, xgb_acc])
best_models = [lr_model, rf_model, xgb_model]
best_names = ['Logistic Regression', 'Random Forest', 'XGBoost']
best_model = best_models[best_idx]
best_name = best_names[best_idx]
print(f'\nBest model: {best_name} (Accuracy: {results.iloc[best_idx]["Accuracy"]:.4f})')

In [ ]:
# Save best model, scaler, and encoders
import os
os.makedirs('model', exist_ok=True)
joblib.dump(best_model, 'model/student_performance_model.pkl')
joblib.dump(scaler, 'model/scaler.pkl')
joblib.dump(le_target, 'model/label_encoder_target.pkl')
joblib.dump(label_encoders, 'model/label_encoders.pkl')
joblib.dump(feature_cols, 'model/feature_cols.pkl')
print('Trained model and artifacts saved to model/')

In [ ]:
# Confusion matrix visualization (best model)
best_cm = [lr_cm, rf_cm, xgb_cm][best_idx]
sns.heatmap(best_cm, annot=True, fmt='d', cmap='Blues', xticklabels=le_target.classes_, yticklabels=le_target.classes_)
plt.title(f'Confusion Matrix - {best_name}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('model/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Confusion matrix saved to model/confusion_matrix.png')